# Function Algorithms - Step 0 (Preparation)

Miguel Angel Ramírez

08.11.2023

This notebook aims to prepare orthoimages for the function experiments between PlanetScope and Sentinel-2.

This notebook:
- Read the orthoimage and mask them according to the zone of interest.
- Reproject the orthoimage if they do not have the same reference system.
- Get the output image which corresponds to the injection (high + medium)
- Obtain high spatial images and medium resolution images resampled for the experiments.
- Average resamplig which are being used in fusion methods.
- Estimated PS SWIR for only fusion purpose based on lineal regression

### Importation of libraries and definition of functions.

In [1]:
import fiona
import rasterio
import rasterio.mask
import geopandas as gpd
import os
import numpy as np
import pyproj
import pandas as pd
import statsmodels.api as sm
import shutil
import time

from osgeo import gdal, osr
from os import walk
from rasterio.warp import reproject, Resampling
from rasterio.mask import mask
from shapely.geometry import Polygon
from rasterio.features import geometry_mask
from shapely.geometry import shape
from shapely.geometry import MultiPolygon
from rasterio.warp import reproject, Resampling
from shapely.geometry import shape, mapping, Polygon
from shapely.ops import unary_union
from fiona.crs import from_epsg
from rasterio.warp import calculate_default_transform, reproject, Resampling
from osgeo import gdal, ogr
import geopandas as gpd
from shapely.geometry import Polygon
from osgeo import gdalconst
from scipy.stats import linregress
from rasterio.merge import merge
from rasterio import Affine
from osgeo import gdal

In [2]:
def preparation (src_raster_path,shp_file_path):

    gdf = gpd.read_file(shp_file_path)
    
    with rasterio.open(src_raster_path) as src:
        crs_raster = src.crs
    
    gdf_reproyectado = gdf.to_crs(crs_raster)
    output_raster_path = src_raster_path[:-4] + '_aoi_masked.tif'

    shapes = [gdf_reproyectado.geometry.values[0]]
    
    with rasterio.open(src_raster_path) as src:
        out_image, out_transform = mask(src, shapes, crop=True)
        out_meta = src.meta

    out_meta.update({"driver": "GTiff",
                     "height": out_image.shape[1],
                     "width": out_image.shape[2],
                     "transform": out_transform})
    
    with rasterio.open(output_raster_path, "w", **out_meta) as dest:
        dest.write(out_image)
    
    return output_raster_path

def reproject_and_replace(image_path1, image_path2 , reprojected_image_path):
    # Open the input images
    ds1 = gdal.Open(image_path1, gdal.GA_ReadOnly)
    ds2 = gdal.Open(image_path2, gdal.GA_ReadOnly)

    if ds1 is None or ds2 is None:
        print("Failed to open one or both of the input images.")
        return

    # Get the projection strings
    proj1 = ds1.GetProjection()
    proj2 = ds2.GetProjection()

    # Check if the projections are different
    if proj1 != proj2:
        print("The input images have different projections. Reprojecting the first image to match the second.")

        # Define the target CRS from the second image
        target_crs = ds2.GetProjection()

        # Reproject the first image to the target CRS
        warp = gdal.Warp(reprojected_image_path,ds1,dstSRS=ds2.GetProjection())

        # Rename the reprojected image to the original filename
        #os.rename(reprojected_image_path, image_path1)

        print(f"The first image has been reprojected: {reprojected_image_path}")
    else:
        print("The input images have the same projection. No reprojection is needed.")
    
    # Close the datasets
    ds1 = None
    ds2 = None

def save_bands_as_tiff(input_image, output_folder, rename_method=None):
    # Verificar si la carpeta de salida existe, si no, crearla
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Abrir la imagen de entrada
    ds = gdal.Open(input_image, gdal.GA_ReadOnly)

    if ds is None:
        print(f"Failed to open the input image: {input_image}")
        return

    # Obtener información de la imagen
    num_bands = ds.RasterCount
    width = ds.RasterXSize
    height = ds.RasterYSize

    # Recorrer todas las bandas y guardarlas como archivos TIFF individuales
    for band_num in range(1, num_bands + 1):
        band = ds.GetRasterBand(band_num)

        # Crear un nombre de archivo para la banda
        if rename_method == "avg":
            # Usar el método "avg" para renombrar la banda
            output_filename = os.path.join(output_folder, f"band{band_num}_avg_fusion.tif")
        else:
            output_filename = os.path.join(output_folder, f"band{band_num}.tif")

        # Crear un nuevo conjunto de datos para la banda y escribir los datos
        driver = gdal.GetDriverByName("GTiff")
        band_ds = driver.Create(output_filename, width, height, 1, band.DataType)

        if band_ds is None:
            print(f"Failed to create the output band: {output_filename}")
            continue

        band_data = band.ReadAsArray()
        band_ds.GetRasterBand(1).WriteArray(band_data)

        # Copiar la información geoespacial y proyección de la imagen original
        band_ds.SetGeoTransform(ds.GetGeoTransform())
        band_ds.SetProjection(ds.GetProjection())

        # Cerrar el conjunto de datos de la banda
        band_ds = None

        print(f"Band {band_num} saved as {output_filename}")

    # Cerrar el conjunto de datos de la imagen original
    ds = None

def reescalar_y_reemplazar_carpeta(input_folder, output_folder):
    # Crear la carpeta de salida si no existe
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Obtener la lista de archivos en la carpeta
    archivos = os.listdir(input_folder)

    for archivo in archivos:
        # Comprobar si el archivo es una imagen raster (puedes ajustar la lista de extensiones según tus necesidades)
        if archivo.lower().endswith(('.tif', '.tiff', '.jpg', '.jpeg', '.png')):
            input_path = os.path.join(input_folder, archivo)
            output_path = os.path.join(output_folder, archivo)

            # Abrir la imagen con GDAL
            dataset = gdal.Open(input_path, gdal.GA_ReadOnly)

            # Verificar si el conjunto de datos se abrió correctamente
            if dataset is None:
                print(f"No se pudo abrir {input_path}. Saltando a la siguiente imagen.")
                continue

            # Obtener la cantidad de bandas
            num_bandas = dataset.RasterCount

            # Crear una lista para almacenar las bandas reescaladas
            bandas_reescaladas = []

            for i in range(1, num_bandas + 1):
                # Obtener la banda
                banda = dataset.GetRasterBand(i)

                # Leer los datos de la banda como un array NumPy
                array_imagen = banda.ReadAsArray()

                # Obtener los valores mínimos y máximos de la imagen con precisión de 13 decimales
                min_valor = np.nanmin(array_imagen).astype(np.float64)
                max_valor = np.nanmax(array_imagen).astype(np.float64)

                # Reescalar los valores de la imagen al rango de 0 a 10000 usando Numpy
                array_imagen_reescalado = np.interp(array_imagen, (min_valor, max_valor), (0, 10000)).astype(np.uint16)

                # Almacenar la banda reescalada en la lista
                bandas_reescaladas.append(array_imagen_reescalado)

            # Copiar información del sistema de referencia y transformación geoespacial al nuevo conjunto de datos
            geotransform = dataset.GetGeoTransform()
            projection = dataset.GetProjection()

            # Cerrar el conjunto de datos GDAL
            dataset = None

            # Agregar un breve retraso (puedes ajustar el valor según sea necesario)
            time.sleep(0.1)

            # Crear un nuevo archivo de imagen con GDAL
            driver = gdal.GetDriverByName('GTiff')
            new_dataset = driver.Create(output_path, array_imagen.shape[1], array_imagen.shape[0], num_bandas, banda.DataType)

            # Verificar si se obtuvo la información del sistema de referencia y transformación geoespacial
            if geotransform is not None and projection is not None:
                new_dataset.SetGeoTransform(geotransform)
                new_dataset.SetProjection(projection)

            # Escribir las bandas reescaladas en el nuevo conjunto de datos
            for i in range(num_bandas):
                new_dataset.GetRasterBand(i + 1).WriteArray(bandas_reescaladas[i])

            # Cerrar el nuevo conjunto de datos GDAL
            new_dataset = None


def resample_image(input_image, reference_image, output_path, resampling_method=gdalconst.GRA_Cubic, output_data_type=gdal.GDT_Float32):
    # Abrir la imagen de referencia
    ref_ds = gdal.Open(reference_image, gdalconst.GA_ReadOnly)
    if ref_ds is None:
        print(f"Error al abrir la imagen de referencia: {reference_image}")
        return

    # Obtener la extensión del archivo de referencia
    ref_extension = os.path.splitext(reference_image)[1]

    # Obtener la información de georreferenciación de la imagen de referencia
    ref_geo_transform = ref_ds.GetGeoTransform()
    ref_projection = ref_ds.GetProjection()
    ref_width = ref_ds.RasterXSize
    ref_height = ref_ds.RasterYSize

    # Abrir la imagen de entrada que se va a remuestrear
    input_ds = gdal.Open(input_image, gdalconst.GA_ReadOnly)
    if input_ds is None:
        print(f"Error al abrir la imagen de entrada: {input_image}")
        return

    # Crear una nueva imagen de salida con la misma resolución y proyección que la de referencia
    driver = gdal.GetDriverByName("GTiff")
    output_path = output_path + ref_extension
    output_ds = driver.Create(output_path, ref_width, ref_height, input_ds.RasterCount, output_data_type)
    output_ds.SetGeoTransform(ref_geo_transform)
    output_ds.SetProjection(ref_projection)

    # Realizar el remuestreo para cada banda de la imagen de entrada
    for band_num in range(1, input_ds.RasterCount + 1):
        input_band = input_ds.GetRasterBand(band_num)
        output_band = output_ds.GetRasterBand(band_num)
        gdal.Warp(
            output_ds, input_ds, format="GTiff", outputType=output_data_type, resampleAlg=resampling_method
        )
        print(band_num)

    # Cerrar las imágenes de entrada y salida
    input_ds = None
    output_ds = None

    return output_path

def stack_rasters(input_paths,output_path):
    
    # Verificar si la carpeta de salida existe, si no, crearla
    if not os.path.exists(os.path.dirname(output_path)):
        os.makedirs(os.path.dirname(output_path))

    # Listas para almacenar las bandas de los rasters
    stacked_bands = []

    # Obtener el dtype del primer raster
    first_band_dtype = None

    # Abrir y apilar las bandas de los rasters de entrada
    for path in input_paths:
        with rasterio.open(path) as src:
            for band in src.read():
                if first_band_dtype is None:
                    first_band_dtype = band.dtype
                # Convertir todas las bandas al mismo dtype
                band = band.astype(first_band_dtype)
                stacked_bands.append(band)

    # Crear un nuevo archivo raster con las bandas apiladas
    with rasterio.open(output_path + "S2_inj_fusion.tif", "w", driver="GTiff", width=src.width, height=src.height, count=len(stacked_bands), dtype=stacked_bands[0].dtype, crs=src.crs, transform=src.transform) as dest:
        for i, band in enumerate(stacked_bands, 1):
            dest.write(band, i)

    print(f'Bandas apiladas y guardadas en {output_path}')

def preparation_data(Project_folder, PS_image, S2_image, aoi):
    
    # File definitions
    
    S2_image = Project_folder + S2_image
    PS_image = Project_folder + PS_image
    aoi = Project_folder + aoi
    imagen = gdal.Open(PS_image)
    imagen2 = gdal.Open(S2_image)
    numero_de_bandas = imagen.RasterCount
    numero_de_bandas2 = imagen2.RasterCount
    del imagen
    del imagen2
    
    # Mask for the study area
    
    S2_image_mask = preparation(S2_image,aoi)
    PS_image_mask = preparation(PS_image,aoi)
    
    # Reprojection of orthoimage
    
    S2_image_mask_p = S2_image_mask[:-4] + '_f.tif'
    S2_image_mask_p_f = S2_image_mask_p[:-4] + '_t.tif'
    #S2_image_mask_p_1 = S2_image_mask_p[:-4] + '_avg.tif'
    reproject_and_replace(S2_image_mask, PS_image_mask, S2_image_mask_p)
    os.remove(S2_image_mask)
    #stack_rasters([S2_image_mask_p, PS_image_mask], Project_folder + "fusion_output/")
    #shutil.copy(S2_image_mask_p, Project_folder + "fusion_output/")
    S2_image_mask_p_f = resample_image(S2_image_mask_p, PS_image_mask, S2_image_mask_p_f)
    #S2_image_mask_p_1 = resample_image(S2_image_mask_p, PS_image_mask, S2_image_mask_p_1, resampling_method=gdalconst.GRA_Average)
 
    # Exportation of individual bands
    
    save_bands_as_tiff(PS_image_mask, Project_folder + "PS_bands")
    os.remove(PS_image_mask)
    save_bands_as_tiff(S2_image_mask_p_f, Project_folder + "S2_bands")
    os.remove(S2_image_mask_p_f)
    reescalar_y_reemplazar_carpeta(Project_folder + "S2_bands", Project_folder + "S2_bands_r")
    os.remove(Project_folder + "S2_bands")
    #save_bands_as_tiff(S2_image_mask_p_1, Project_folder + "S2_f", "avg")
    calculate_and_apply_regression(Project_folder + "S2_bands_r", Project_folder + "PS_bands", numero_de_bandas, numero_de_bandas2)
    #os.remove(S2_image_mask_p_1)

def calcular_promedio_de_bandas(imagen1, imagen2, nombre_salida):
    # Abre las dos imágenes de entrada con GDAL
    img1 = gdal.Open(imagen1)
    img2 = gdal.Open(imagen2)

    # Leer los datos de las bandas de las imágenes en forma de matrices NumPy
    datos_banda1 = img1.GetRasterBand(1).ReadAsArray()
    datos_banda2 = img2.GetRasterBand(1).ReadAsArray()

    # Calcular el promedio de las dos bandas
    promedio = (datos_banda1 + datos_banda2) / 2

    # Obtener información de una de las imágenes originales para crear la nueva imagen
    driver = gdal.GetDriverByName('GTiff')
    filas, columnas = promedio.shape
    numero_bandas = 1  # El promedio se almacena en una sola banda

    # Crear una nueva imagen TIFF
    nueva_imagen = driver.Create(nombre_salida, columnas, filas, numero_bandas, gdalconst.GDT_Float32)

    # Escribir el promedio en la nueva imagen
    nueva_imagen.GetRasterBand(1).WriteArray(promedio)

    # Definir la proyección y la geotransformación de la nueva imagen (usando la información de img1)
    nueva_imagen.SetProjection(img1.GetProjection())
    nueva_imagen.SetGeoTransform(img1.GetGeoTransform())

    # Cerrar ambas imágenes
    img1 = None
    img2 = None
    nueva_imagen = None

def calculate_and_apply_regression(input_folder, output_folder, nbands, nbands2):
    if nbands == 8 and nbands2 == 12:
        band4_raster = gdal.Open(os.path.join(input_folder, "band4.tif"))
        band8_raster = gdal.Open(os.path.join(input_folder, "band8.tif"))
        band11_raster = gdal.Open(os.path.join(input_folder, "band11.tif"))
        band12_raster = gdal.Open(os.path.join(input_folder, "band12.tif"))
        output_path_band11 = os.path.join(output_folder, "band11_est.tif")
        output_path_band12 = os.path.join(output_folder, "band12_est.tif")
    elif nbands == 4 and nbands2 == 12:
        band4_raster = gdal.Open(os.path.join(input_folder, "band3.tif"))
        band8_raster = gdal.Open(os.path.join(input_folder, "band4.tif"))  
        band11_raster = gdal.Open(os.path.join(input_folder, "band11.tif"))
        band12_raster = gdal.Open(os.path.join(input_folder, "band12.tif"))
        output_path_band11 = os.path.join(output_folder, "band11_est.tif")
        output_path_band12 = os.path.join(output_folder, "band12_est.tif")
    elif nbands == 4 and nbands2 == 11:
        band4_raster = gdal.Open(os.path.join(input_folder, "band3.tif"))
        band8_raster = gdal.Open(os.path.join(input_folder, "band4.tif"))  
        band11_raster = gdal.Open(os.path.join(input_folder, "band10.tif"))
        band12_raster = gdal.Open(os.path.join(input_folder, "band11.tif"))
        output_path_band11 = os.path.join(output_folder, "band10_est.tif")
        output_path_band12 = os.path.join(output_folder, "band11_est.tif")
    elif nbands == 5 and nbands2 == 11:
        band4_raster = gdal.Open(os.path.join(input_folder, "band3.tif"))
        band8_raster = gdal.Open(os.path.join(input_folder, "band5.tif"))  
        band11_raster = gdal.Open(os.path.join(input_folder, "band10.tif"))
        band12_raster = gdal.Open(os.path.join(input_folder, "band11.tif"))
        output_path_band11 = os.path.join(output_folder, "band10_est.tif")
        output_path_band12 = os.path.join(output_folder, "band11_est.tif")
    elif nbands == 5 and nbands2 == 7:
        band4_raster = gdal.Open(os.path.join(input_folder, "band3.tif"))
        band8_raster = gdal.Open(os.path.join(input_folder, "band4.tif"))  
        band11_raster = gdal.Open(os.path.join(input_folder, "band5.tif"))
        band12_raster = gdal.Open(os.path.join(input_folder, "band6.tif"))
        output_path_band11 = os.path.join(output_folder, "band5_est.tif")
        output_path_band12 = os.path.join(output_folder, "band7_est.tif")
        
    # Realiza la regresión lineal para band11.tif
    independent_data_band11 = band4_raster.ReadAsArray().flatten()
    independent_data_band11 = np.column_stack((independent_data_band11, band8_raster.ReadAsArray().flatten()))

    dependent_data_band11 = band11_raster.ReadAsArray().flatten()

    # Filtra valores infinitos o NaN
    valid_mask_band11 = np.isfinite(independent_data_band11).all(axis=1)
    independent_data_band11 = independent_data_band11[valid_mask_band11]
    dependent_data_band11 = dependent_data_band11[valid_mask_band11]

    model_band11 = sm.OLS(dependent_data_band11, sm.add_constant(independent_data_band11)).fit()

    # Realiza la regresión lineal para band12.tif
    independent_data_band12 = band4_raster.ReadAsArray().flatten()
    independent_data_band12 = np.column_stack((independent_data_band12, band8_raster.ReadAsArray().flatten()))

    dependent_data_band12 = band12_raster.ReadAsArray().flatten()

    # Filtra valores infinitos o NaN
    valid_mask_band12 = np.isfinite(independent_data_band12).all(axis=1)
    independent_data_band12 = independent_data_band12[valid_mask_band12]
    dependent_data_band12 = dependent_data_band12[valid_mask_band12]

    model_band12 = sm.OLS(dependent_data_band12, sm.add_constant(independent_data_band12)).fit()

    # Aplica el modelo a los rasters en la carpeta de salida
    if nbands == 8:
        independent_data_output = np.column_stack((gdal.Open(os.path.join(output_folder, "band6.tif")).ReadAsArray().flatten(), gdal.Open(os.path.join(output_folder, "band8.tif")).ReadAsArray().flatten()))
    else:
        output_path_avg34 = os.path.join(output_folder, "band34_avg.tif")
        calcular_promedio_de_bandas(os.path.join(output_folder, "band3.tif"), os.path.join(output_folder, "band4.tif"), output_path_avg34)
        independent_data_output = np.column_stack((gdal.Open(os.path.join(output_folder, "band3.tif")).ReadAsArray().flatten(), gdal.Open(os.path.join(output_folder, "band4.tif")).ReadAsArray().flatten()))
    
    predicted_data_band11 = model_band11.predict(sm.add_constant(independent_data_output))
    predicted_data_band12 = model_band12.predict(sm.add_constant(independent_data_output))

    # Aplica la normalización para ajustar el rango a [0, 10000]
    min_value = 0
    max_value = 10000
    predicted_data_band11 = np.interp(predicted_data_band11, (predicted_data_band11.min(), predicted_data_band11.max()), (min_value, max_value))
    predicted_data_band12 = np.interp(predicted_data_band12, (predicted_data_band12.min(), predicted_data_band12.max()), (min_value, max_value))

    # Convierte las predicciones en matrices 2D
    predicted_data_band11 = predicted_data_band11.reshape(band11_raster.ReadAsArray().shape)
    predicted_data_band12 = predicted_data_band12.reshape(band12_raster.ReadAsArray().shape)

    # Guarda los rasters estimados en la carpeta de salida
    driver = gdal.GetDriverByName('GTiff')
    output_raster_band11 = driver.Create(output_path_band11, band11_raster.RasterXSize, band11_raster.RasterYSize, 1, gdal.GDT_Float32)
    output_raster_band11.GetRasterBand(1).WriteArray(predicted_data_band11)

    # Establecer el sistema de referencia para los rasters estimados
    proj_info = band11_raster.GetProjection()
    output_raster_band11.SetProjection(proj_info)
    output_raster_band11.SetGeoTransform(band11_raster.GetGeoTransform())  # Establecer la información de coordenadas
    output_raster_band11.FlushCache()

    output_raster_band12 = driver.Create(output_path_band12, band12_raster.RasterXSize, band12_raster.RasterYSize, 1, gdal.GDT_Float32)
    output_raster_band12.GetRasterBand(1).WriteArray(predicted_data_band12)

    # Establecer el sistema de referencia para los rasters estimados
    proj_info = band12_raster.GetProjection()
    output_raster_band12.SetProjection(proj_info)
    output_raster_band12.SetGeoTransform(band12_raster.GetGeoTransform())  # Establecer la información de coordenadas
    output_raster_band12.FlushCache()

The function of preparation_data is to do all the processes. Then, it is highlighted to define the function parameters, which are.
- The folder of the project where the orthoimages are
- The name of PlanetScope/RapidEye orthoimage
- The name of Sentinel-2/Landsat-8 orthoimage
- The study zone in shapefile format

In [ ]:
preparation_data('E:\PLANETSCOPE\MONTES 2011/',"Montes 2011.tif","finalj.tif", "montes_z.shp")

The output are:
- The bands of each orthoimage in the system reference of PlanetScope orthoimage in the folders of PS_bands and S2_bands

In [4]:
Project_folder= 'E:\PLANETSCOPE\MONTES 2011/'

calculate_and_apply_regression(Project_folder + "S2_bands_r", Project_folder + "PS_bands", 5,7)